### The Müller-Brown Potential
Динамика Ланжевена — широко используемый метод, реализованный в таких мощных программных пакетах, как NAMD, Amber и LAMMPS, которые способны моделировать молекулярные системы, содержащие сотни тысяч атомов. Прежде чем перейти к системам с сотнями тысяч и более степенями свободы, рассмотрим гораздо более простую систему с двумя пространственными измерениями — потенциал Мюллера-Броуна. Этот «простой» двумерный потенциал представлен следующим уравнением для потенциальной энергии как функции положения в измерениях x и y (U(x, y)):

\begin{equation}
\begin{split}
U(x, y) &= -170 \exp \left(11 (x+0.5) (y-1.5)-6.5 (x+0.5)^2-6.5 (y-1.5)^2\right)\\
            &   +15 \exp \left(0.6 (x+1) (y-1)+0.7(x+1)^2+0.7 (y-1)^2\right)\\
            &   -100 e^{-x^2-10 (y-0.5)^2}-200 e^{-(x-1)^2-10 y^2}\\
\end{split}
\end{equation}

Лучший способ получить некоторое представление о нём — взглянуть на график потенциала. Видео 2, представленное ниже, — один из способов визуализации потенциала [Мюллера-Броуна](https://sci-hub.ru/10.1007/BF00547608).

In [14]:
# Демонстрационная анимация будет построена ниже из рассчитанной траектории.
print('Потенциал Мюллера — Брауна: моделирование 40 частиц')

Потенциал Мюллера — Брауна: моделирование 40 частиц


__Видео 2:__ Здесь представлена визуализация данных потенциала Мюллера-Брауна, созданная мной с помощью Python и Pymol.

Представьте, что вы хаотично бьёте футбольным мячом по поверхности внутри чаши, показанной на видео, и вы поймёте, что такое моделирование двумерной частицы, движущейся под действием потенциала Мюллера-Брауна. Обратите внимание: если вы ударите мяч и позволите ему катиться до полной остановки, то появятся три области устойчивости, в которых он потенциально может остановиться. Эти области устойчивости особенно заметны на начальном изображении, поскольку внутри большей чаши, созданной потенциалом, чётко видны три отдельных подобласти. Благодаря этим особенностям, наличию всего двух пространственных измерений (энергия может быть представлена в виде третьего измерения) и природе множества экспоненциальных функций, составляющих потенциал Мюллера-Брауна, он стал признанным эталоном для тестирования методологий молекулярного моделирования. Это также отлично подходит для тех, кто хочет узнать, что происходит «под капотом» моделирования молекулярной динамики!

### Моделирование траекторий на основе потенциала Мюллера-Броуна

Теперь, когда мы имеем представление о динамике Ланжевена и потенциале Мюллера-Брауна, давайте соединим их вместе и начнём моделирование! Приведённая ниже ячейка кода импортирует все необходимые библиотеки, сам код Ланжевена (langevin_2D.py) и определяет все переменные, используемые в этом моделировании. Если вы хотите запустить новое моделирование с другими параметрами, следует иметь в виду несколько моментов. Во-первых, эти параметры были подвергнуты существенной настройке для получения траекторий, сходящихся к желаемой температуре (подробнее об этом позже) за разумное время моделирования. Использование параметров, которые каким-либо образом «ломают» код, может быть хорошим способом обучения, но имейте в виду, что такое может произойти. Ещё один важный момент: существует булева переменная _runFresh_, которая определяет, будет ли этот блокнот выполняться с использованием ранее сгенерированных данных (False) или с использованием новой траектории выполнения (True). Обязательно измените её на True, если хотите изменить параметры в словаре _param_ (иначе код проигнорирует новые параметры).

In [15]:
import numpy as np
import matplotlib.pylab as plt 
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import langevin_2D as langevin
import writeXYZ as xyz
import plottingFunctions as myPlt
import pickle
import timeit

runFresh = True # Run a new trajectory?
# Энергия и температура в приведённых единицах, k_B = 1.
# Первичный пик температуры: потенциальная энергия переходит в кинетическую;
# затем термостат отводит избыток энергии и приводит систему к заданной температуре.

In [16]:
start_time = timeit.default_timer()

trajectoryFileName = "oldTrajectory.pickle"

params = {
        'natoms': 40, #how many particles we want
        'temp': .15, #temperature, .5 works well, too hot makes Müller-Brown unstable! was .25
        'mass': 0.001, #mass of the particle, was .001
        'atomType': "H", #type of atom for xyz file
        'relax': .5e-13, #dissipation constant for thermostat, .5e-13 works well
        'dt': 1.25e-17, # 1e-15 is too large, 
        'steps': 20000, #number of steps to run
        'freq': 50, #frequency to save positions
        'box': ((-1.7, 1.2), (-.7, 2.2)), #bounds in x and y direction
        'initPos': (-.75, .6), #initial position to center cloud of particles
        }

if runFresh:
    %matplotlib inline
    trajectory = langevin.run(**params)
    with open(trajectoryFileName, 'wb') as oldTrajectory:
        pickle.dump(trajectory, oldTrajectory)
else:
    with open(trajectoryFileName, 'rb') as oldTrajectory:
        trajectory = pickle.load(oldTrajectory)
    

In [ ]:
myPlt.plotTemperatures("temperatures.txt", params['dt'])

Рисунок 1: Этот график показывает зависимость температуры моделирования от времени. Обратите внимание, что моделирование сходится к температуре, заданной в моделировании (при условии использования исходного набора параметров).

### Анализ результатов моделирования
#### Температура
Первое, на что следует обратить внимание, — это изменение температуры моделирования в зависимости от времени, показанное на рисунке 1 выше. Обратите внимание, как температура начинается с нуля, достигает пика около 2,8 К, а затем снижается до заданной температуры 0,5 К. Помните, что скорость движения атомов и температура системы, которую они составляют, взаимосвязаны (чем быстрее частицы, тем выше температура). Учитывая начальное положение траекторий в моделировании, можете ли вы объяснить, почему температура демонстрирует поведение, показанное на рисунке 1? Молекулярное моделирование с тысячами атомов также всегда требует некоторого времени для достижения заданной температуры. Такие молекулы, как белки и нуклеиновые кислоты, одни из наиболее распространённых молекулярных систем, изучаемых с помощью молекулярной динамики, также часто имеют несколько областей в своих энергетических «ландшафтах» (т.е. поверхностях потенциальной энергии). Ситуация для этих систем осложняется тем, что вместо двух пространственных измерений _x_ и _y_ со сложными экспоненциальными зависимостями нам приходится учитывать сотни тысяч измерений! Надеюсь, это иллюстрирует, почему потенциал Мюллера-Брауна является столь ценной тестовой системой для проверки методов молекулярного моделирования перед их применением даже к малым молекулярным системам. Даже дипептид аланина, самая маленькая белковоподобная молекула, часто используемая в дальнейшем при разработке методов молекулярной динамики, содержит $22$ атома, что даёт $3\times22 - 6 = 60$ пространственных измерений!

#### Построение траекторий
А теперь давайте немного поиграем с траекториями! Из словаря _param_ Python, приведённого выше, следует, что 100 частиц (_natoms_ = 100) подвергаются динамике Ланжевена в потенциале Мюллера-Броуна. Нажмите кнопку воспроизведения на изображении ниже, и вы увидите, как траектория меняется во времени. Обратите внимание, что траектории начинаются на «более высоких уровнях» (энергетически говоря) и в конечном итоге достигают трёх минимумов, где они оказываются захваченными. Многие принципы, которые вы можете наблюдать и интуитивно понимать при визуальном использовании потенциала Мюллера-Броуна, можно напрямую применить к гораздо более сложным молекулярным системам. Подумайте о том, как температура системы изменяется со временем по сравнению с движением частиц. Рассмотрите такие химические понятия, как переходные состояния и энергии активации, и сравните их с «холмами», которые траектории должны преодолеть, чтобы пройти между локальными минимумами.

In [23]:
cartoon = myPlt.getTrajMovie(trajectory, params['box'])
cartoon.save('traj.gif', writer='pillow', fps=14, dpi=70)
HTML(cartoon.to_jshtml())

KeyboardInterrupt: 

### Анимация наших траекторий Ланжевена на потенциале Мюллера-Броуна
Здесь показаны результаты расчёта динамики Ланжевена на потенциале Мюллера-Броуна. Если вы хотите запустить новую траекторию и/или изменить параметры моделирования, обязательно установите runFresh = True.

![SegmentLocal](traj.gif "segment")

__Видео 3:__ Анимированная визуализация данных, показывающая 40 траекторий, подчиняющихся динамике Ланжевена на потенциале Мюллера-Броуна, представленном контурным графиком.
#### Обратите внимание:
Вы можете посмотреть эту же анимацию в формате, позволяющем управлять воспроизведением, в нижней части этого блокнота Jupyter.

#### Ячейка ниже используется для создания .xyz файлов:

In [19]:
myPlt.writeTrajXYZ(trajectory, 'traj-hydrogen-3D.xyz')
myPlt.getSurfaceForPymol(params['box'], 100, 100, 'surface.xyz')

Вы могли заметить, что помимо построения графика результатов моделирования в этом блокноте Jupyter, результаты также экспортировались в двух разных форматах. Один из файлов, экспортированных функцией _langevin.getMovie()_ (вызываемой в ячейке ниже), — это файл _traj.gif_. Хотя это, вероятно, не очень подходящий GIF-файл для мемов, он обладает той же функциональностью, что и GIF-файлы, которыми вы можете поделиться в социальных сетях, то есть вы можете открыть его в своем любимом браузере и воспроизвести как фильм. langevin_2D.py также генерирует несколько других выходных файлов: _surface.xyz_, _traj-hydrogen-3D.xyz_ и _temperatures.txt_. Файл _temperatures.txt_ — это простой текстовый файл, содержащий температуру на каждом временном шаге моделирования, который позже загружается обратно в Python для построения графика. Два файла с расширениями _.xyz_ были созданы для загрузки в программное обеспечение для визуализации молекулярной динамики, такое как Pymol или VMD. Ниже в __Видео 4__ показана 3D-визуализация этих же траекторий вместе с некоторыми весьма мелодраматичными комментариями. То, что здесь сделано, не является типичным использованием Pymol, поскольку сначала выведена сетка атомов водорода, которые будут лежать на потенциале Мюллера-Брауна, с использованием оси z для представления значения энергии. Нет никаких причин делать их атомами водорода, разве что пришлось обмануть Pymol, заставив его думать, что это какой-то атом! Траектории также были выведены как для атомов водорода по той же причине. Позже мы собираемся научиться создавать подобные видео с использованием реальных молекул (мелодраматичные комментарии будут необязательны).

In [20]:
# Отображаем анимацию вместо отсутствующего внешнего видео.
HTML(cartoon.to_jshtml())

__Видео 4:__ Это видео демонстрирует более красивую трёхмерную анимированную визуализацию данных, демонстрирующую 100 траекторий, подчиняющихся динамике Ланжевена на потенциале Мюллера-Броуна. Конечно, это немного мелодраматично, но, надеюсь, поможет вам прочувствовать динамику. :-)

# Домашнее задание:
1) Используйте анимацию ниже, чтобы изучить, как меняется температура в зависимости от движения траекторий. Помните, что вы можете ускорить или замедлить анимацию с помощью кнопок «+» и «-», остановить её в любом месте, двигаться по одному шагу и т. д. Своими словами объясните, что происходит по мере приближения системы к равновесной температуре.

In [21]:
%%capture
tempWithTraj = myPlt.getMovieWithTemp(trajectory, params["box"], "temperatures.txt")

In [22]:
HTML(tempWithTraj.to_jshtml())

KeyboardInterrupt: 

In [ ]:
elapsed = timeit.default_timer() - start_time
print("This notebook ran in", "%.2f" % round(elapsed/60, 2),"minutes.")